In [ ]:
import sys
sys.path.append('..')  # Add parent directory to path
import csv

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset as HFDataset

from peft import LoraConfig, get_peft_model
from utils import load_rwku_data, prepare_tokenized_dataset, evaluate_model, evaluate_neighbours

# Pick the best available device: CUDA (NVIDIA, e.g. Windows/Linux) -> MPS (Apple) -> CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def empty_device_cache():
    """Free cached GPU memory for whichever backend is active."""
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()

In [2]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

SUBJECT   = "Donald Trump"

In [ ]:

print(f"Loading model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)

model = model.to(DEVICE)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj", "gate_proj","up_proj","down_proj"], # Layers which will be unlearned
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)

print("Number of parameters for training:")
peft_model.print_trainable_parameters()

In [4]:
# Load data

person_train, questions_forget, keywords_forget, questions_retain, keywords_retain = load_rwku_data(SUBJECT)
tokenized_forget = prepare_tokenized_dataset(person_train, tokenizer)

tokenized_forget = tokenized_forget.map(lambda x: {"labels": x["input_ids"]})
tokenized_forget.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("\nDatasets ready\n")

Questions for forgetting test: 20
Questions for general knowledge test: 30
Texts for training (unlearning): 226
Data is ready

Datasets ready



In [5]:
print("\n------------------------ BEFORE UNLEARNING EVALUATION ------------------------\n")

print("\nBASELINE EFFICACY TEST")
acc_forget_before = evaluate_model(model, tokenizer, questions_forget, keywords_forget, DEVICE)

print("\nBASELINE NEIGHBOURS TEST")
acc_retain_before = evaluate_model(model, tokenizer, questions_retain, keywords_retain, DEVICE)

print("=" * 60)
print("  SUMMARY")
print("=" * 60)
print(f"  Method              : base model")
print(f"  Subject             : {SUBJECT}")
print(f"  Efficacy (forget %) : {acc_forget_before:.2f}%")
print(f"  Utility  (retain %) : {acc_retain_before:.2f}%")
print("=" * 60)


------------------------ BEFORE UNLEARNING EVALUATION ------------------------


BASELINE EFFICACY TEST
Starting model testing

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th
donald john trump served as the 45th president of the united states'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Donald Trump co-produced and hosted the reality television series ___
Expected: 'the apprentice'
Model generated: 'the apprentice

the sentence would be completed as: "from 2004 to 2'
Result: PASSED

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th
donald john trump served as the 45th president of the united states'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Tru

In [6]:
class GradientAscentTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss = outputs.loss

        # Unlearning - multiply error by -1
        unlearning_loss = -1.0 * loss  # Instead of minimizing the error (learning), we maximize it (unlearning)

        return (unlearning_loss, outputs) if return_outputs else unlearning_loss


In [7]:
# Different hyperparameters

GRID = [
    {"lr": 1e-4, "max_steps": 30},
    {"lr": 1e-4, "max_steps": 100},
    {"lr": 1e-4, "max_steps": 300},
    {"lr": 3e-5, "max_steps": 30},
    {"lr": 3e-5, "max_steps": 100},
    {"lr": 3e-5, "max_steps": 300},
    {"lr": 5e-6, "max_steps": 30},
    {"lr": 5e-6, "max_steps": 100},
    {"lr": 5e-6, "max_steps": 300},
]

In [ ]:
csv_path = "./ga_unlearning_grid_results_Qwen2.5-3B.csv"

with open(csv_path, "w", newline="") as f:
    csv.DictWriter(f, fieldnames=[
        "model", "subject", "lr", "max_steps",
        "efficacy_before", "efficacy_after",
        "neighbours_before", "neighbours_after",
    ]).writeheader()

for config in GRID:
    lr, max_steps = config["lr"], config["max_steps"]
    print(f"\n{'='*60}")
    print(f"Config: lr={lr}  max_steps={max_steps}")
    print(f"{'='*60}")

    fresh_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16).to(DEVICE)
    fresh_peft  = get_peft_model(fresh_model, LoraConfig(
        r=8, 
        lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, 
        bias="none", 
        task_type="CAUSAL_LM",
    ))

    training_args = TrainingArguments(
        output_dir=f"./ga_unlearning_lr{lr}_max_steps{max_steps}_qwen2.5-3B",
        per_device_train_batch_size=1,      
        gradient_accumulation_steps=2,      
        learning_rate=lr,
        max_steps=max_steps,
        logging_steps=2,       
        optim="adamw_torch"          
    )

    trainer = GradientAscentTrainer(
        model=fresh_peft,
        args=training_args,
        train_dataset=tokenized_forget,
    )

    trainer.train()
    print("Unlearning finished")


    print("\n------------------------ AFTER UNLEARNING EVALUATION ------------------------\n")

    print("UNLEARNING EFFICACY TEST")
    acc_forget = evaluate_model(fresh_peft, tokenizer, questions_forget, keywords_forget, DEVICE)

    print()
    print("UNLEARNING UTILITY TEST (Knowledge Retention)")
    acc_retain = evaluate_model(fresh_peft, tokenizer, questions_retain, keywords_retain, DEVICE)

    print("\n" + "=" * 60)
    print("  SUMMARY")
    print("=" * 60)
    print(f"  Method              : Gradient Ascent (Pure Unlearning)")
    print(f"  Subject             : {SUBJECT}")
    print(f"  Efficacy (forget %) : {acc_forget_before:.2f}% -> {acc_forget:.2f}%  (lower is better)")
    print(f"  Utility  (retain %) : {acc_retain_before:.2f}% -> {acc_retain:.2f}%  (higher is better)")
    print("=" * 60)
    

    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "model", "subject", "lr", "max_steps",
            "efficacy_before", "efficacy_after",
            "neighbours_before", "neighbours_after",
        ])
        writer.writerow({
            "model": MODEL_ID, "subject": SUBJECT,
            "lr": lr, "max_steps": max_steps,
            "efficacy_before":   f"{acc_forget_before:.1f}",
            "efficacy_after":    f"{acc_forget:.1f}",
            "neighbours_before": f"{acc_retain_before:.1f}",
            "neighbours_after":  f"{acc_retain:.1f}",
        })

    del fresh_model, fresh_peft, trainer
    empty_device_cache()

print(f"\nAll done. Results saved to {csv_path}.")